# LULC projection (2 m) — ensemble CA + four scenarios

# This notebook implements a workflow for projecting future land-use/land-cover (LULC) patterns under two scenarios using historical spatial data. It processes and analyzes LULC datasets to generate scenario-based projections, enabling the assessment of potential future landscape changes and their spatial distribution.


## 1. Imports

In [ ]:
from pathlib import Path
import json
import warnings

import geopandas as gpd
import joblib
import numpy as np
from pyproj import CRS
import rasterio
from rasterio.features import rasterize
from rasterio.vrt import WarpedVRT
from rasterio.warp import Resampling
from scipy.ndimage import convolve, generic_filter, label as nd_label
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    confusion_matrix, f1_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
print("Libraries loaded successfully.")

## 2. Inputs, outputs, settings

In [ ]:
DATA_DIR = Path("data")  # Place the required input datasets under this folder.

LULC_2003 = DATA_DIR / "2m/Projected_ready/resampl2m/lulc2m.tif"
LULC_2009 = DATA_DIR / "2m/Projected_ready/resampl2m/lulc2m2009.tif.tif"
LULC_2020 = DATA_DIR / "2m/Projected_ready/resampl2m/lulc2m2020.tif"

BASIN     = DATA_DIR / "2m/Projected_ready/Gidra basin boundary/Gidra_basin_boundary_WMmap50_updated.shp"
PROTECTED = DATA_DIR / "2m/Projected_ready/Clipped_Protected areas/Clipped.shp"

DEM   = DATA_DIR / "2m/Projected_ready/resampl2m/dem2020.tif"
SLOPE = DATA_DIR / "2m/Projected_ready/resampl2m/slope2m.tif"

DIST_ROAD_2001 = DATA_DIR / "2m/Projected_ready/resampl2m/dist_road2001.tif"
DIST_ROAD_2009 = DATA_DIR / "2m/Projected_ready/resampl2m/dist_road2009.tif"
DIST_ROAD_2023 = DATA_DIR / "2m/Projected_ready/resampl2m/dist_road2023.tif"
DIST_RIVER     = DATA_DIR / "2m/Projected_ready/resampl2m/dis_river2m.tif"
DIST_RAILWAY   = DATA_DIR / "2m/Projected_ready/resampl2m/railway2m.tif"

POP_2000 = DATA_DIR / "2m/Projected_ready/resampl2m/Pop2000.tif"
POP_2010 = DATA_DIR / "2m/Projected_ready/resampl2m/Pop2010.tif"
POP_2020 = DATA_DIR / "2m/Projected_ready/resampl2m/Pop2020.tif"

SOIL_RASTER = DATA_DIR / "2m/Projected_ready/resampl2m/Soil.tif"
SOILS_SHP   = None
SOIL_FIELD  = "soils_en"

ORTHO_2003 = DATA_DIR / "Data/Ortophotos/resampled orto2003/ortho2003_basin_2m.tif"
ORTHO_2020 = DATA_DIR / "Data/Ortophotos/Orto2020.res2m/orto2020res2m.tif"

NDVI_2001 = DATA_DIR / "2m/Projected_ready/indices/NDVI_2001_2m_clipped.tif"
NDWI_2001 = DATA_DIR / "2m/Projected_ready/indices/NDWI_2001_2m_clipped.tif"
NDBI_2001 = DATA_DIR / "2m/Projected_ready/indices/NDBI_2001_2m_clipped.tif"
NDPI_2001 = DATA_DIR / "2m/Projected_ready/indices/NDPI_2001_2m_clipped.tif"

NDVI_2009 = DATA_DIR / "2m/Projected_ready/indices/NDVI_2009_2m_clipped.tif"
NDWI_2009 = DATA_DIR / "2m/Projected_ready/indices/NDWI_2009_2m_clipped.tif"
NDBI_2009 = DATA_DIR / "2m/Projected_ready/indices/NDBI_2009_2m_clipped.tif"
NDPI_2009 = DATA_DIR / "2m/Projected_ready/indices/NDPI_2009_2m_clipped.tif"

NDVI_2020 = DATA_DIR / "2m/Projected_ready/indices/NDVI_2020_2m_clipped.tif"
NDWI_2020 = DATA_DIR / "2m/Projected_ready/indices/NDWI_2020_2m_clipped.tif"
NDBI_2020 = DATA_DIR / "2m/Projected_ready/indices/NDBI_2020_2m_clipped.tif"
NDPI_2020 = DATA_DIR / "2m/Projected_ready/indices/NDPI_2020_2m_clipped.tif"

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESOLUTION_M = 2.0
TARGET_YEARS = [2030, 2040, 2050]
RANDOM_SEED = 42
SAMPLE_PER_TRANSITION = 150_000
PROBA_BATCH = 500_000
BACKYARD_PER_BUILDING = 0.35
BUILDING_MIN_GROWTH = 1.08
MAJORITY_FILTER_SIZE = 5
NEIGHBOURHOOD_WEIGHT = 1.4
NEIGHBOURHOOD_SIZE = 5
IMMUTABLE_CLASSES = {4, 10}
LINEAR_CLASSES = {4, 10}
MIN_ROAD_COMPONENT = 50

CLASS_NAMES = {
    0: "NoData", 1: "Arable land", 2: "Building", 3: "Built-up area",
    4: "Channel", 5: "Forest", 6: "Glade", 7: "Grassland",
    8: "Material/waste dump", 9: "Orchard/garden", 10: "Road (asphalt)",
    11: "Shrub", 12: "Urban greenery", 13: "Vineyard", 14: "Water body",
    15: "Backyard garden",
}

SCENARIOS = {
    "business_as_usual": {
        "description": "Historical transition rates; modest building growth",
        "class_mult": {},
        "building_growth": 1.08,
        "protect_nature": False,
    },
    "policy_driven": {
        "description": "Moderate urban growth; limit loss of arable & forest",
        "class_mult": {2: 1.15, 3: 1.10, 15: 1.12, 1: 0.95, 5: 1.05, 7: 1.02},
        "building_growth": 1.12,
        "protect_nature": False,
    },
    "conservation": {
        "description": "Protect forest, grassland, water; slow urbanisation",
        "class_mult": {2: 0.90, 3: 0.92, 15: 0.95, 5: 1.15, 7: 1.10, 14: 1.05, 1: 0.98},
        "building_growth": 1.02,
        "protect_nature": True,
    },
    "urban_expansion": {
        "description": "Strong building & built-up growth from arable/grassland",
        "class_mult": {2: 1.45, 3: 1.35, 15: 1.30, 1: 0.85, 7: 0.88, 5: 0.95},
        "building_growth": 1.35,
        "protect_nature": False,
    },
}

print("OUT_DIR:", OUT_DIR)
print("Scenarios:", list(SCENARIOS.keys()))
print("Target years:", TARGET_YEARS)

## 3. Helper functions

In [ ]:
def require_file(value, label):
    if value is None:
        return
    if not Path(value).exists():
        raise FileNotFoundError(f"{label} does not exist: {value}")


def normalize_crs(crs_like):
    if crs_like is None:
        return None
    try:
        c = CRS.from_user_input(crs_like)
    except Exception:
        c = CRS.from_user_input(str(crs_like))
    try:
        epsg = c.to_epsg()
        if epsg is not None:
            return CRS.from_epsg(epsg)
    except Exception:
        pass
    text = (c.to_wkt() or str(c)).upper()
    if any(k in text for k in ("S-JTSK", "KROVAK", "5514", "JTSK")):
        return CRS.from_epsg(5514)
    return c


def same_crs(a, b):
    if a is None or b is None:
        return False
    a, b = normalize_crs(a), normalize_crs(b)
    try:
        if a.to_epsg() is not None and a.to_epsg() == b.to_epsg():
            return True
    except Exception:
        pass
    try:
        return a == b
    except Exception:
        return str(a) == str(b)


def grid_from_raster(path):
    with rasterio.open(path) as src:
        crs = normalize_crs(src.crs)
        transform = src.transform
        shape = (src.height, src.width)
        res = abs(src.transform.a)
    return crs, transform, shape, res


def basin_mask_on_grid(basin_path, crs, transform, shape):
    crs = normalize_crs(crs)
    layer = gpd.read_file(basin_path)
    if layer.crs is None:
        raise ValueError(f"Basin has no CRS: {basin_path}")
    layer_crs = normalize_crs(layer.crs)
    if not same_crs(layer_crs, crs):
        layer = layer.set_crs(layer_crs, allow_override=True).to_crs(crs)
    else:
        layer = layer.set_crs(crs, allow_override=True)
    mask = rasterize(
        ((geom, 1) for geom in layer.geometry if geom is not None and not geom.is_empty),
        out_shape=shape, transform=transform, fill=0, dtype="uint8",
    )
    return mask.astype(bool)


def rasterize_polygons(path, crs, transform, shape, value_field=None):
    crs = normalize_crs(crs)
    layer = gpd.read_file(path)
    if layer.crs is None:
        raise ValueError(f"Polygon layer has no CRS: {path}")
    layer_crs = normalize_crs(layer.crs)
    if not same_crs(layer_crs, crs):
        layer = layer.set_crs(layer_crs, allow_override=True).to_crs(crs)
    else:
        layer = layer.set_crs(crs, allow_override=True)
    if value_field is None:
        shapes_iter = ((geom, 1) for geom in layer.geometry if geom is not None and not geom.is_empty)
    else:
        shapes_iter = (
            (geom, int(v))
            for geom, v in zip(layer.geometry, layer[value_field])
            if geom is not None and not geom.is_empty
        )
    return rasterize(shapes_iter, out_shape=shape, transform=transform, fill=0, dtype="int32")


def read_raster_on_grid(path, crs, transform, shape, resampling=Resampling.bilinear, band=1):
    crs = normalize_crs(crs)
    result = np.full(shape, np.nan, dtype="float32")
    with rasterio.open(path) as source:
        source_crs = normalize_crs(source.crs)
        b = min(band, source.count)
        with WarpedVRT(
            source, src_crs=source_crs, crs=crs, transform=transform,
            width=shape[1], height=shape[0], src_nodata=source.nodata,
            nodata=np.nan, resampling=resampling,
        ) as warped:
            warped.read(b, out=result)
    return result


def load_lulc(path, crs, transform, shape):
    arr = read_raster_on_grid(str(path), crs, transform, shape, resampling=Resampling.nearest)
    out = np.zeros(shape, dtype="int32")
    finite = np.isfinite(arr)
    out[finite] = np.rint(arr[finite]).astype("int32")
    return out


def load_continuous(path, crs, transform, shape):
    return np.nan_to_num(
        read_raster_on_grid(str(path), crs, transform, shape, resampling=Resampling.bilinear),
        nan=0.0,
    ).astype("float32")


def load_ortho_brightness(path, crs, transform, shape):
    with rasterio.open(path) as src:
        n = src.count
    bands = [
        read_raster_on_grid(str(path), crs, transform, shape, resampling=Resampling.bilinear, band=b)
        for b in range(1, min(n, 3) + 1)
    ]
    return np.nanmean(np.stack(bands, axis=0), axis=0).astype("float32")


def slope_degrees(dem, res_m):
    gy, gx = np.gradient(dem.astype("float64"), res_m, res_m)
    return np.degrees(np.arctan(np.hypot(gx, gy))).astype("float32")


def rasterize_categorical(shp_path, crs, transform, shape, field):
    layer = gpd.read_file(shp_path)
    layer = layer.set_crs(normalize_crs(layer.crs), allow_override=True)
    if not same_crs(layer.crs, crs):
        layer = layer.to_crs(normalize_crs(crs))
    cats = sorted(layer[field].astype(str).unique())
    legend = {i + 1: c for i, c in enumerate(cats)}
    inv = {c: i + 1 for i, c in enumerate(cats)}
    shapes_iter = (
        (geom, inv[str(val)])
        for geom, val in zip(layer.geometry, layer[field])
        if geom is not None and not geom.is_empty
    )
    arr = rasterize(shapes_iter, out_shape=shape, transform=transform, fill=0, dtype="int32")
    return arr, legend


def values_at_valid(arr, valid):
    return arr[valid].astype("int32", copy=False)


def build_features(lulc_arr, drivers, valid):
    n = int(np.count_nonzero(valid))
    n_drv = len(drivers)
    X = np.empty((n, 1 + n_drv), dtype="float32")
    X[:, 0] = lulc_arr[valid].astype("float32")
    for i, d in enumerate(drivers):
        X[:, i + 1] = d[valid].astype("float32")
    return X


def sample_training(X, y, max_samples, seed):
    n = len(y)
    if n <= max_samples:
        return X, y
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=max_samples, replace=False)
    return X[idx], y[idx]


def fit_models(X, y, classes):
    class_list = [int(c) for c in classes]
    label_to_idx = {c: i for i, c in enumerate(class_list)}
    y_idx = np.array([label_to_idx[int(v)] for v in y], dtype="int32")

    models = {}

    rf = RandomForestClassifier(
        n_estimators=200, max_depth=28, min_samples_leaf=2,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_SEED,
    )
    rf.fit(X, y_idx)
    models["CA_RF"] = rf

    xgb = XGBClassifier(
        n_estimators=250, max_depth=8, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        objective="multi:softprob", num_class=len(class_list),
        tree_method="hist", random_state=RANDOM_SEED, n_jobs=-1, verbosity=0,
    )
    xgb.fit(X, y_idx)
    models["CA_XGBoost"] = xgb

    mlp = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(96, 48), max_iter=200,
            early_stopping=True, random_state=RANDOM_SEED,
        ),
    )
    mlp.fit(X, y_idx)
    models["CA_MLP"] = mlp

    for m in models.values():
        m._class_list = class_list
    return models


def class_probabilities(models, X, class_count):
    n = X.shape[0]
    out = np.zeros((n, class_count), dtype="float32")
    n_models = len(models)
    for start in range(0, n, PROBA_BATCH):
        end = min(start + PROBA_BATCH, n)
        batch = X[start:end]
        acc = np.zeros((end - start, class_count), dtype="float32")
        for model in models.values():
            p = model.predict_proba(batch).astype("float32")
            if p.shape[1] == class_count:
                acc += p
            else:
                for j, lab in enumerate(getattr(model, "classes_", range(p.shape[1]))):
                    acc[:, int(lab)] += p[:, j]
        out[start:end] = acc / n_models
    return out


def transition_matrix(src, dst, classes, valid, immutable=None):
    immutable = set(immutable or [])
    n = len(classes)
    idx = {int(c): i for i, c in enumerate(classes)}
    counts = np.zeros((n, n), dtype="float64")
    s = src[valid].ravel()
    d = dst[valid].ravel()
    for a, b in zip(s, d):
        a, b = int(a), int(b)
        if a not in idx or b not in idx:
            continue
        if a in immutable:
            counts[idx[a], idx[a]] += 1
        else:
            counts[idx[a], idx[b]] += 1
    P = counts.copy()
    row_sum = P.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    P = P / row_sum
    return counts, P


def demand_counts(current_lulc, P, classes, years_ahead, building_growth=1.08, class_mult=None):
    class_mult = class_mult or {}
    n = len(classes)
    idx = {int(c): i for i, c in enumerate(classes)}
    counts = np.array([np.count_nonzero(current_lulc == c) for c in classes], dtype="float64")
    n_steps = max(1, int(round(years_ahead / 10.0)))
    for _ in range(n_steps):
        counts = counts @ P
    if 2 in idx:
        base = np.count_nonzero(current_lulc == 2)
        counts[idx[2]] = max(counts[idx[2]], base * (building_growth ** n_steps))
    for c, m in class_mult.items():
        if int(c) in idx:
            counts[idx[int(c)]] *= float(m)
    if 2 in idx and 15 in idx:
        b0 = np.count_nonzero(current_lulc == 2)
        b1 = counts[idx[2]]
        extra_b = max(0.0, b1 - b0)
        counts[idx[15]] = max(
            counts[idx[15]],
            np.count_nonzero(current_lulc == 15) + BACKYARD_PER_BUILDING * extra_b,
        )
    total = int(np.count_nonzero(current_lulc != 0))
    counts = np.maximum(counts, 0)
    if counts.sum() <= 0:
        counts = np.array([np.count_nonzero(current_lulc == c) for c in classes], dtype="float64")
    counts = counts / counts.sum() * total
    out = np.floor(counts).astype("int64")
    rem = total - int(out.sum())
    order = np.argsort(-(counts - out))
    for k in range(max(0, rem)):
        out[order[k % n]] += 1
    return out


def allocate_ca(current, suitability_valid, classes, target_counts,
                basin_mask, protected, immutable, seed, valid):
    """
    Robust CA allocation.

    suitability_valid : (n_valid, n_classes) from class_probabilities
    valid             : boolean mask used to BUILD suitability_valid
                        (must have exactly n_valid True cells)

    Suitability is scattered onto a full raster by spatial position,
    so there is never a row-index mismatch with a different mask.
    """
    rng = np.random.default_rng(seed)
    out = current.copy()
    valid = np.asarray(valid, dtype=bool)

    class_list = [int(c) for c in classes]
    n_cls = len(class_list)
    idx = {c: i for i, c in enumerate(class_list)}

    n_true = int(np.count_nonzero(valid))
    if suitability_valid.shape[0] != n_true:
        raise ValueError(
            f"suitability rows ({suitability_valid.shape[0]}) != "
            f"True cells in valid ({n_true}). "
            "Pass the exact mask used in build_features / class_probabilities."
        )
    if suitability_valid.shape[1] != n_cls:
        raise ValueError(
            f"suitability cols ({suitability_valid.shape[1]}) != n_classes ({n_cls})"
        )

    # Scatter 1-D probabilities onto full 2-D maps (spatial lookup, no row index issues)
    suit_map = np.zeros((n_cls,) + current.shape, dtype="float32")
    for i in range(n_cls):
        suit_map[i][valid] = suitability_valid[:, i]

    def neighbourhood_fraction(arr, class_id, size=NEIGHBOURHOOD_SIZE):
        kernel = np.ones((size, size), dtype="float32")
        binary = (arr == class_id).astype("float32")
        return convolve(binary, kernel, mode="nearest") / float(size * size)

    # Only reassign cells that are in `valid` (have suitability)
    # and are currently non-zero inside the basin
    work = valid & basin_mask & (current != 0)
    work_rc = np.column_stack(np.where(work))

    cur_counts = {c: int(np.count_nonzero(out == c)) for c in class_list}
    deficit, surplus = {}, {}
    for c, tgt in zip(class_list, target_counts):
        diff = int(tgt) - cur_counts[c]
        if diff > 0:
            deficit[c] = diff
        elif diff < 0:
            surplus[c] = -diff

    if not deficit or not surplus:
        return out

    for dest, need in sorted(deficit.items(), key=lambda x: -x[1]):
        if need <= 0:
            continue
        dest_i = idx[dest]
        neigh = neighbourhood_fraction(out, dest, NEIGHBOURHOOD_SIZE)

        scores = []
        for r, c in work_rc:
            src = int(out[r, c])
            if src not in surplus or surplus[src] <= 0:
                continue
            if src in immutable or dest in immutable:
                continue
            if protected[r, c]:
                continue
            s = max(float(suit_map[dest_i, r, c]), 1e-7)
            score = np.log(s) + NEIGHBOURHOOD_WEIGHT * float(neigh[r, c])
            score += rng.normal(0, 1e-8)
            scores.append((score, r, c, src))

        if not scores:
            continue
        scores.sort(key=lambda t: -t[0])
        taken = 0
        for score, r, c, src in scores:
            if taken >= need:
                break
            if surplus.get(src, 0) <= 0:
                continue
            if int(out[r, c]) != src:
                continue
            out[r, c] = dest
            surplus[src] -= 1
            taken += 1

    return out


def clean_map(arr):
    out = arr.copy()
    linear_mask = np.isin(arr, list(LINEAR_CLASSES))
    linear_vals = arr[linear_mask]

    def _maj(win):
        win = win.astype("int32")
        vals, cnts = np.unique(win, return_counts=True)
        return vals[np.argmax(cnts)]

    filtered = generic_filter(
        out.astype("float64"), _maj, size=MAJORITY_FILTER_SIZE, mode="nearest"
    )
    out = filtered.astype(arr.dtype)
    out[linear_mask] = linear_vals

    if 10 in np.unique(out):
        road = out == 10
        labeled, nlab = nd_label(road)
        for lab in range(1, nlab + 1):
            comp = labeled == lab
            if int(comp.sum()) < MIN_ROAD_COMPONENT:
                ys, xs = np.where(comp)
                for y, x in zip(ys, xs):
                    y0, y1 = max(0, y - 2), min(out.shape[0], y + 3)
                    x0, x1 = max(0, x - 2), min(out.shape[1], x + 3)
                    patch = out[y0:y1, x0:x1]
                    patch = patch[patch != 10]
                    if patch.size:
                        vals, cnts = np.unique(patch, return_counts=True)
                        out[y, x] = vals[np.argmax(cnts)]
    return out


def save_map(path, arr, crs, transform):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    profile = {
        "driver": "GTiff", "height": arr.shape[0], "width": arr.shape[1],
        "count": 1, "dtype": "int32", "crs": crs, "transform": transform,
        "compress": "lzw", "tiled": True, "nodata": 0,
    }
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(arr.astype("int32"), 1)
    print(f"  saved {path}")


def save_legend_csv(path, classes):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    lines = ["class_code,class_name"]
    for c in classes:
        lines.append(f"{int(c)},{CLASS_NAMES.get(int(c), int(c))}")
    path.write_text("\n".join(lines), encoding="utf-8")


def evaluate(pred, truth, classes, valid):
    y_true = truth[valid]
    y_pred = pred[valid]
    labels = [int(c) for c in classes]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "kappa": float(cohen_kappa_score(y_true, y_pred, labels=labels)),
        "confusion_matrix": cm.tolist(),
        "labels": labels,
    }


def change_assessments(origin, simulated, observed, classes, valid):
    o = origin[valid]
    s = simulated[valid]
    t = observed[valid]
    obs_change = o != t
    sim_change = o != s
    H = int(np.count_nonzero(obs_change & sim_change & (s == t)))
    M = int(np.count_nonzero(obs_change & ~sim_change))
    F = int(np.count_nonzero(~obs_change & sim_change))
    W = int(np.count_nonzero(obs_change & sim_change & (s != t)))
    denom = H + M + F + W
    fom = H / denom if denom else 0.0
    labels = [int(c) for c in classes]
    r = np.array([np.count_nonzero(t == c) for c in labels], dtype="float64")
    p = np.array([np.count_nonzero(s == c) for c in labels], dtype="float64")
    r /= max(r.sum(), 1)
    p /= max(p.sum(), 1)
    Q = 0.5 * np.abs(r - p).sum()
    D = float(np.mean(s != t))
    Dalloc = max(0.0, D - Q)
    return {
        "H": H, "M": M, "F": F, "W": W,
        "figure_of_merit": float(fom),
        "quantity_disagreement": float(Q),
        "allocation_disagreement": float(Dalloc),
        "total_disagreement": float(D),
    }


print("Helpers ready.")

## 4. Check inputs exist

In [ ]:
for label, path in [
    ("LULC_2003", LULC_2003), ("LULC_2009", LULC_2009), ("LULC_2020", LULC_2020),
    ("BASIN", BASIN), ("PROTECTED", PROTECTED),
    ("DEM", DEM), ("DIST_ROAD_2001", DIST_ROAD_2001), ("DIST_ROAD_2009", DIST_ROAD_2009),
    ("DIST_ROAD_2023", DIST_ROAD_2023), ("DIST_RIVER", DIST_RIVER), ("DIST_RAILWAY", DIST_RAILWAY),
    ("POP_2000", POP_2000), ("POP_2010", POP_2010), ("POP_2020", POP_2020),
    ("SOIL_RASTER", SOIL_RASTER),
    ("ORTHO_2003", ORTHO_2003), ("ORTHO_2020", ORTHO_2020),
    ("NDVI_2001", NDVI_2001), ("NDVI_2009", NDVI_2009), ("NDVI_2020", NDVI_2020),
]:
    require_file(path, label)
if SLOPE:
    require_file(SLOPE, "SLOPE")
print("All required paths exist.")

## 5. Load data on common 2 m grid

In [ ]:
crs, transform, shape, res = grid_from_raster(LULC_2020)
print(f"Master grid from LULC_2020: shape={shape}, res={res}, CRS={crs}")
if abs(res - RESOLUTION_M) > 0.15:
    print(f"WARNING: LULC_2020 resolution is {res} m, settings say {RESOLUTION_M} m")

basin_mask = basin_mask_on_grid(BASIN, crs, transform, shape)
print(f"Basin cells: {np.count_nonzero(basin_mask):,}")

print("Loading LULC ...")
lulc = {
    2001: load_lulc(LULC_2003, crs, transform, shape),
    2010: load_lulc(LULC_2009, crs, transform, shape),
    2020: load_lulc(LULC_2020, crs, transform, shape),
}
for y, a in lulc.items():
    print(f"  era {y}: classes={sorted(int(c) for c in np.unique(a) if c)}")

protected_raster = rasterize_polygons(PROTECTED, crs, transform, shape) > 0
print(f"Protected cells: {np.count_nonzero(protected_raster & basin_mask):,}")

classes = np.array(sorted(
    (set(np.unique(lulc[2001])) | set(np.unique(lulc[2010])) | set(np.unique(lulc[2020]))) - {0}
), dtype="int32")
class_count = len(classes)
print("Classes:", [CLASS_NAMES.get(int(c), c) for c in classes])

print("DEM / slope ...")
dem = load_continuous(DEM, crs, transform, shape)
if SLOPE and Path(SLOPE).exists():
    slope = load_continuous(SLOPE, crs, transform, shape)
else:
    slope = slope_degrees(dem, RESOLUTION_M)

print("Distances ...")
dist_roads_2001 = load_continuous(DIST_ROAD_2001, crs, transform, shape)
dist_roads_2009 = load_continuous(DIST_ROAD_2009, crs, transform, shape)
dist_roads_2023 = load_continuous(DIST_ROAD_2023, crs, transform, shape)
dist_rivers = load_continuous(DIST_RIVER, crs, transform, shape)
dist_rail = load_continuous(DIST_RAILWAY, crs, transform, shape)

print("Population ...")
pop_2000 = load_continuous(POP_2000, crs, transform, shape)
pop_2010 = load_continuous(POP_2010, crs, transform, shape)
pop_2020 = load_continuous(POP_2020, crs, transform, shape)

print("Soil ...")
if SOIL_RASTER and Path(SOIL_RASTER).exists():
    soil_driver = np.nan_to_num(
        read_raster_on_grid(SOIL_RASTER, crs, transform, shape, resampling=Resampling.nearest),
        nan=0.0,
    ).astype("float32")
else:
    soil_arr, soil_legend = rasterize_categorical(SOILS_SHP, crs, transform, shape, SOIL_FIELD)
    soil_driver = soil_arr.astype("float32")
    (OUT_DIR / "soil_class_legend.json").write_text(json.dumps(soil_legend, indent=2), encoding="utf-8")

print("Orthophotos (brightness) ...")
ortho_2003 = load_ortho_brightness(ORTHO_2003, crs, transform, shape)
ortho_2020 = load_ortho_brightness(ORTHO_2020, crs, transform, shape)

print("Spectral indices ...")
ndvi_2001 = load_continuous(NDVI_2001, crs, transform, shape)
ndwi_2001 = load_continuous(NDWI_2001, crs, transform, shape)
ndbi_2001 = load_continuous(NDBI_2001, crs, transform, shape)
ndpi_2001 = load_continuous(NDPI_2001, crs, transform, shape)
ndvi_2009 = load_continuous(NDVI_2009, crs, transform, shape)
ndwi_2009 = load_continuous(NDWI_2009, crs, transform, shape)
ndbi_2009 = load_continuous(NDBI_2009, crs, transform, shape)
ndpi_2009 = load_continuous(NDPI_2009, crs, transform, shape)
ndvi_2020 = load_continuous(NDVI_2020, crs, transform, shape)
ndwi_2020 = load_continuous(NDWI_2020, crs, transform, shape)
ndbi_2020 = load_continuous(NDBI_2020, crs, transform, shape)
ndpi_2020 = load_continuous(NDPI_2020, crs, transform, shape)

drivers_2001 = [
    dem, slope, dist_roads_2001, dist_rivers, dist_rail, pop_2000, soil_driver,
    ortho_2003, ndvi_2001, ndwi_2001, ndbi_2001, ndpi_2001,
]
drivers_2010 = [
    dem, slope, dist_roads_2009, dist_rivers, dist_rail, pop_2010, soil_driver,
    ortho_2003, ndvi_2009, ndwi_2009, ndbi_2009, ndpi_2009,
]
drivers_2020 = [
    dem, slope, dist_roads_2023, dist_rivers, dist_rail, pop_2020, soil_driver,
    ortho_2020, ndvi_2020, ndwi_2020, ndbi_2020, ndpi_2020,
]

# Single consistent mask for training, probabilities, and allocation
valid = basin_mask & (lulc[2001] != 0) & (lulc[2010] != 0) & (lulc[2020] != 0)
print(f"Valid cells: {np.count_nonzero(valid):,}")
print(f"Drivers per period: {len(drivers_2001)}")

## 6. Hindcast validation (2010 → 2020)

In [ ]:
print("=== Hindcast 2010 → 2020 ===")

x_early = build_features(lulc[2001], drivers_2001, valid)
y_early = values_at_valid(lulc[2010], valid)
x_early, y_early = sample_training(x_early, y_early, SAMPLE_PER_TRANSITION, RANDOM_SEED)
print(f"Training samples (2001→2010): {len(y_early):,}")

hindcast_models = fit_models(x_early, y_early, classes)

x_late = build_features(lulc[2010], drivers_2010, valid)
probabilities = class_probabilities(hindcast_models, x_late, class_count)
print(f"Probability matrix: {probabilities.shape}  (must equal valid count {np.count_nonzero(valid)})")

_, late_markov = transition_matrix(lulc[2010], lulc[2020], classes, valid)
late_demand = np.array([np.count_nonzero(lulc[2020] == v) for v in classes])

# valid=valid is REQUIRED — same mask used for probabilities
hindcast_raw = allocate_ca(
    current=lulc[2010],
    suitability_valid=probabilities,
    classes=classes,
    target_counts=late_demand,
    basin_mask=basin_mask,
    protected=protected_raster,
    immutable=IMMUTABLE_CLASSES,
    seed=RANDOM_SEED,
    valid=valid,
)

hc_dir = OUT_DIR / "hindcast"
save_map(hc_dir / "hindcast_2020_raw.tif", hindcast_raw, crs, transform)
hindcast = clean_map(hindcast_raw)
save_map(hc_dir / "hindcast_2020.tif", hindcast, crs, transform)

metrics = evaluate(hindcast, lulc[2020], classes, valid)
persistence_metrics = evaluate(lulc[2010], lulc[2020], classes, valid)
hindcast_assessments = change_assessments(lulc[2010], hindcast, lulc[2020], classes, valid)
persistence_assessments = change_assessments(lulc[2010], lulc[2010], lulc[2020], classes, valid)
metrics["persistence_baseline"] = persistence_metrics
metrics["assessments"] = {
    "hindcast_2010_to_2020": hindcast_assessments,
    "persistence_baseline_2010": persistence_assessments,
}
(hc_dir / "hindcast_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
save_legend_csv(hc_dir / "class_legend.csv", classes)

print("Hindcast metrics:")
for k in ("accuracy", "balanced_accuracy", "macro_f1", "kappa"):
    print(f"  {k}: {metrics[k]:.4f}")
print(f"  FoM: {hindcast_assessments['figure_of_merit']:.4f}")

## 7. Final models + four scenario projections

In [ ]:
print("=== Final models ===")
x_late_full = build_features(lulc[2010], drivers_2010, valid)
y_late_full = values_at_valid(lulc[2020], valid)
x_late_sample, y_late_sample = sample_training(
    x_late_full, y_late_full, SAMPLE_PER_TRANSITION, RANDOM_SEED + 1
)
x_final = np.vstack([x_early, x_late_sample])
y_final = np.concatenate([y_early, y_late_sample])
print(f"Final training samples: {len(y_final):,}")

final_models = fit_models(x_final, y_final, classes)

# Future mask: basin + 2020 non-zero (may differ slightly from `valid`)
future_valid = basin_mask & (lulc[2020] != 0)
x_future = build_features(lulc[2020], drivers_2020, future_valid)
future_probability = class_probabilities(final_models, x_future, class_count)
print(f"Future probability matrix: {future_probability.shape}  (valid={np.count_nonzero(future_valid)})")

NATURE_CLASSES = {5, 14}
all_scenario_metrics = {}

for sname, scfg in SCENARIOS.items():
    print(f"\n########## SCENARIO: {sname} ##########")
    print(scfg["description"])
    sdir = OUT_DIR / "scenarios" / sname
    sdir.mkdir(parents=True, exist_ok=True)

    immutable = set(IMMUTABLE_CLASSES)
    if scfg.get("protect_nature"):
        immutable = immutable | NATURE_CLASSES

    _, scen_markov = transition_matrix(
        lulc[2010], lulc[2020], classes, valid, immutable=immutable
    )

    scen_metrics = {"description": scfg["description"], "years": {}}

    for target_year in TARGET_YEARS:
        years_ahead = target_year - 2020
        target_counts = demand_counts(
            lulc[2020], scen_markov, classes, years_ahead,
            building_growth=scfg.get("building_growth", BUILDING_MIN_GROWTH),
            class_mult=scfg.get("class_mult", {}),
        )
        cur = np.array([np.count_nonzero(lulc[2020] == c) for c in classes])
        need = int(np.abs(target_counts.astype("int64") - cur).sum() // 2)
        print(f"\n  {target_year} ({years_ahead} y) — ~{need:,} cells to reassign")
        for c, n, c0 in zip(classes, target_counts, cur):
            if int(c) in (2, 3, 15, 5, 1) or n != c0:
                print(f"    {CLASS_NAMES.get(int(c), c)}: {c0:,} → {n:,}")

        projected_raw = allocate_ca(
            current=lulc[2020],
            suitability_valid=future_probability,
            classes=classes,
            target_counts=target_counts,
            basin_mask=basin_mask,
            protected=protected_raster,
            immutable=immutable,
            seed=RANDOM_SEED,
            valid=future_valid,
        )
        save_map(sdir / f"lulc_{target_year}_raw.tif", projected_raw, crs, transform)
        projected = clean_map(projected_raw)
        save_map(sdir / f"lulc_{target_year}.tif", projected, crs, transform)

        summary = {}
        for c, n, c0 in zip(classes, target_counts, cur):
            summary[CLASS_NAMES.get(int(c), str(int(c)))] = {
                "count_2020": int(c0),
                "count_projected": int(np.count_nonzero(projected == c)),
                "target_demand": int(n),
            }
        scen_metrics["years"][str(target_year)] = summary
        print(f"    Saved {sdir / f'lulc_{target_year}.tif'}")

    save_legend_csv(sdir / "class_legend.csv", classes)
    (sdir / "scenario_summary.json").write_text(
        json.dumps(scen_metrics, indent=2), encoding="utf-8"
    )
    all_scenario_metrics[sname] = scen_metrics

(OUT_DIR / "all_scenarios_summary.json").write_text(
    json.dumps({
        "hindcast_metrics_file": str(OUT_DIR / "hindcast" / "hindcast_metrics.json"),
        "scenarios": {k: v["description"] for k, v in SCENARIOS.items()},
    }, indent=2),
    encoding="utf-8",
)

for name, model in final_models.items():
    joblib.dump(model, OUT_DIR / f"{name}.joblib")
    print(f"Saved {name}.joblib")

print(f"\n======== ALL DONE ========")
print(f"Outputs root: {OUT_DIR}")